## Imports

In [ ]:
import numpy as np
import torch
from torch import nn

from shared.constants import DEVICE
from shared.mat_reader import MatReader
from shared.normalization import ZScoreNormalizer
from architectures.binary_ce.dataset import BinaryCEDataset

## Dataset

In [ ]:
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

Datapoint = tuple[torch.Tensor, int, str]  # (patch tensor, class label, patient id)

## Model

In [ ]:
import pretrained_microscopy_models as pmm
import torch.utils.model_zoo as model_zoo
model = torch.hub.load("pytorch/vision:v0.10.0", "resnet50", weights=None)
url = pmm.util.get_pretrained_microscopynet_url("resnet50", "micronet")
model.load_state_dict(model_zoo.load_url(url, map_location=DEVICE))

# from torchvision import models
# weights = models.ResNet18_Weights.DEFAULT
# model = models.resnet18(weights=weights)

# weights = models.ResNet50_Weights.DEFAULT
# model = models.resnet50(weights=weights)

# Strip the classification head to expose the 2048-dimensional pooling layer
model.fc = nn.Identity()

# Load Meta's DINOv2 (Vision Transformer)
# 'dinov2_vits14' is the small version. You can also try 'vitb14', 'vitl14', or 'vitg14'
# model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
# model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
# Note: DINOv2 is a feature extractor. It doesn't have a classification head by default.
# You will pass your images through it and train a linear classifier or MLP on the output features.
# features = model(your_input_tensor)

# from transformers import AutoImageProcessor, AutoModel
# Load Phikon (Pathology-specific ViT)
# processor = AutoImageProcessor.from_pretrained("owkin/phikon")
# model = AutoModel.from_pretrained("owkin/phikon")
# Example usage:
# inputs = processor(images, return_tensors="pt")
# outputs = model(**inputs)
# features = outputs.last_hidden_state

# from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
# from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
# Example 1: ConvNeXt (Often outperforms ResNet50 with similar parameters)
# convnext_weights = ConvNeXt_Tiny_Weights.DEFAULT
# model = convnext_tiny(weights=convnext_weights)
# Example 2: EfficientNetV2 (Optimized for training speed and parameter efficiency)
# effnet_weights = EfficientNet_V2_S_Weights.DEFAULT
# model = efficientnet_v2_s(weights=effnet_weights)

# import timm
# Swin Transformers are great for high-resolution microscopy because they compute 
# attention locally in "windows", preserving fine-grained cellular details.
# model = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=2)
# Or easily try a ResNeXt, which groups convolutions and often beats standard ResNet
# model = timm.create_model('resnext50_32x4d', pretrained=True, num_classes=2)

# from transformers import AutoImageProcessor, AutoModel
# import os
# os.environ["HF_TOKEN"] = "TOKEN HERE"
# # Load phikon-v2
# processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
# model = AutoModel.from_pretrained("owkin/phikon-v2")

model.eval()
model.to(DEVICE)

## StratifiedGroupKFolds

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import GroupKFold
from torch.utils.data import WeightedRandomSampler, DataLoader

from shared.utils import get_n_splits
from shared.constants import CLASS_NAMES

# mat_reader = MatReader("/home/student/lampe-cnn/full_images/")
mat_reader = MatReader("/Users/james/GitHub/lampe/lampe_dataset/Full images/")
n_splits = min(10, get_n_splits(mat_reader))
n_splits = 10
print(f"# splits: {n_splits}")

# No Stratification, No Augmentation, Unbalanced (micronet_svm recreation)
# Uses class_weights='balanced' in SVM params
all_indices = list(range(mat_reader.get_num_fovs()))
# dataset = BinaryCEDataset(mat_reader, eff_fov_indices=all_indices, train=True)
dataset = BinaryCEDataset(mat_reader, eff_fov_indices=all_indices, train=False)

# assumes class label ordering
# class_weights = 1.0 / np.bincount(mat_reader.class_labels[all_indices])
# sample_weights = class_weights[mat_reader.class_labels[all_indices]]

# sampler = WeightedRandomSampler(
#     weights=sample_weights, 
#     num_samples=len(sample_weights) * 2, 
#     replacement=True
# )

# Batch size can be relatively large since we aren't storing gradients
dataloader = DataLoader(dataset, batch_size=32)
# dataloader = DataLoader(dataset, batch_size=32, sampler=sampler)

all_features = []
all_labels = []
all_groups = []

with torch.inference_mode():
# with torch.no_grad():
    model.eval()
    for batch in dataloader:
        images, labels, patient_ids = batch

        # owkin/phikon-v2
        # inputs = processor(images, return_tensors="pt").to(DEVICE)
        # inputs = processor(images, do_rescale=False, do_normalize=False, return_tensors="pt").to(DEVICE)
        # outputs = model(**inputs)
        # features = outputs.last_hidden_state[:, 0, :]

        features = model(images.to(DEVICE))
        # features = model(images.to(DEVICE)[:, :, :504, :504]) # for DINOv2
        all_features.append(features.numpy(force=True))
        # all_features.append(features.detach().cpu().numpy())
        all_labels.append(labels)
        all_groups.append(patient_ids)

X = np.vstack(all_features)
y = np.concatenate(all_labels)
groups = np.concatenate(all_groups)

gkf = GroupKFold(n_splits=n_splits)

# Stratified, Augmented, Balanced
# sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
# X, y, groups = mat_reader.images, mat_reader.class_labels, mat_reader.patient_ids

fold_accuracies: list[float] = []
all_y_true: list[int] = []
all_y_pred: list[int] = []


svm_classifier = SVC(
    kernel="poly", class_weight="balanced", decision_function_shape="ovo"
    # kernel="poly", class_weight={0: 1, 1: 5}, decision_function_shape="ovo"
) # calling ".fit()" resets the SVM

for fold, (train_indices, test_indices) in enumerate(gkf.split(X, y, groups=groups)):
# for fold, (train_indices, test_indices) in enumerate(sgkf.split(X, y, groups=groups)):
    
    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    # Scale SVM inputs. Fit ONLY on training data to avoid data leakage.
    # TODO: investigate interaction with robust_minmax
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # PCA dimensionality reduction. Fit ONLY on training data.
    pca = PCA(n_components=0.95, svd_solver="full")
    X_train_scaled = pca.fit_transform(X_train_scaled)
    X_test_scaled = pca.transform(X_test_scaled)
    print(f"Fold {fold+1}: PCA reduced features to {pca.n_components_} dims")

    # Train and Predict
    svm_classifier.fit(X_train_scaled, y_train)
    y_pred = svm_classifier.predict(X_test_scaled)

    # Record Metrics
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(float(acc))
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

    print(f"Fold {fold+1} Accuracy: {acc * 100:.2f}%")
    cm = confusion_matrix(y_test, y_pred, labels=list(range(len(CLASS_NAMES))))
    print(f"Confusion Matrix (Fold {fold+1}):")
    header = "          " + "  ".join(f"{name:>10}" for name in CLASS_NAMES)
    print(header)
    for i, row in enumerate(cm):
        row_str = "  ".join(f"{v:>10}" for v in row)
        print(f"{CLASS_NAMES[i]:>10}  {row_str}")

print("\n--- Final Results ---")
print(
    f"Mean CV Accuracy: {np.mean(fold_accuracies) * 100:.2f}% ± {np.std(fold_accuracies) * 100:.2f}%"
)

print("\nGlobal Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=CLASS_NAMES))     

print(fold_accuracies)